프로젝트 루트 확인

In [3]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())


프로젝트 루트: c:\dev\team4-shopping-dashboard
데이터 폴더: c:\dev\team4-shopping-dashboard\data\raw
데이터 폴더 존재: True


필수 파일 확인

In [4]:
required_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

for file_name in required_files:

    file_path = data_dir / file_name
    print(file_name, file_path.exists(), file_path.stat().st_size if file_path.exists() else 0)

customers.csv True 51742
products.csv True 17553
orders.csv True 278533
order_items.csv True 335212


데이터 불러옴

In [5]:
import pandas as pd

print(data_dir / "customers.csv")
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

c:\dev\team4-shopping-dashboard\data\raw\customers.csv


In [6]:
datasets = {

    "customers": customers,

    "products": products,

    "orders": orders,

    "order_items": order_items,

}


기본 구조 파악

In [7]:
for name, df in datasets.items():
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print(df.dtypes)


=== customers ===
shape: (1200, 6)
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

=== products ===
shape: (300, 4)
product_id      int64
product_name      str
category          str
price           int64
dtype: object

=== orders ===
shape: (6000, 5)
order_id          int64
customer_id       int64
order_date          str
payment_method      str
order_status        str
dtype: object

=== order_items ===
shape: (14603, 5)
order_item_id    int64
order_id         int64
product_id       int64
quantity         int64
unit_price       int64
dtype: object


중복 행 + 주요 키 중복 확인

In [8]:
pk_map = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

for name, df in datasets.items():
    pk = pk_map[name]
    print(f"\n=== {name} ===")
    print("전체 행 중 완전 중복 행:", df.duplicated().sum())
    print(f"주요 키({pk}) 중복:", df[pk].duplicated().sum())


=== customers ===
전체 행 중 완전 중복 행: 0
주요 키(customer_id) 중복: 0

=== products ===
전체 행 중 완전 중복 행: 0
주요 키(product_id) 중복: 0

=== orders ===
전체 행 중 완전 중복 행: 0
주요 키(order_id) 중복: 0

=== order_items ===
전체 행 중 완전 중복 행: 0
주요 키(order_item_id) 중복: 0


결측치 + 중복 + PK 

In [9]:
pk_map = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

for name, df in datasets.items():
    pk = pk_map[name]
    print(f"\n=== {name} ===")
    print("결측치 합계:", df.isna().sum().sum())
    print("전체 중복행:", df.duplicated().sum())
    print(f"{pk} 중복:", df[pk].duplicated().sum())


=== customers ===
결측치 합계: 0
전체 중복행: 0
customer_id 중복: 0

=== products ===
결측치 합계: 0
전체 중복행: 0
product_id 중복: 0

=== orders ===
결측치 합계: 0
전체 중복행: 0
order_id 중복: 0

=== order_items ===
결측치 합계: 0
전체 중복행: 0
order_item_id 중복: 0


범주형 값 확인

In [10]:
print(orders['order_status'].unique())
print(orders['payment_method'].unique())
print(products['category'].unique())

<StringArray>
['배송중', '환불', '배송완료', '결제완료', '배송준비', '취소']
Length: 6, dtype: str
<StringArray>
['간편결제', '신용카드', '계좌이체', '휴대폰결제', '무통장입금']
Length: 5, dtype: str
<StringArray>
['전자기기', '생활가전', '패션', '뷰티', '식품', '도서', '스포츠', '반려동물', '문구', '홈인테리어']
Length: 10, dtype: str


날짜 컬럼을 실제 datetime으로 변환

In [11]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

# 변환 실패(NaT) 건수 확인 — 0건이어야 정상
print(customers['signup_date'].isna().sum())
print(orders['order_date'].isna().sum())

0
0


수치 컬럼 이상치(음수, 0 이하) 확인

In [12]:
print(customers['age'].describe())
print(products['price'].describe())
print(order_items[['quantity', 'unit_price']].describe())

# 음수/0 이하만 따로 걸러보기
order_items[(order_items['quantity'] <= 0) | (order_items['unit_price'] <= 0)]
products[products['price'] <= 0]
customers[(customers['age'] <= 0) | (customers['age'] > 100)]

count    1200.000000
mean       36.408333
std        10.607247
min        18.000000
25%        29.000000
50%        36.000000
75%        43.000000
max        69.000000
Name: age, dtype: float64
count       300.000000
mean      82423.333333
std       70271.742887
min        4000.000000
25%       30750.000000
50%       62500.000000
75%      115000.000000
max      371000.000000
Name: price, dtype: float64
           quantity     unit_price
count  14603.000000   14603.000000
mean       1.615832   77884.400466
std        0.948504   65289.672212
min        1.000000    3200.000000
25%        1.000000   29700.000000
50%        1.000000   59000.000000
75%        2.000000  107000.000000
max        5.000000  371000.000000


,customer_id,name,gender,age,city,signup_date


범주형 값 표기 오염 확인

In [13]:
for col in ['gender', 'city']:
    print(col, ':', customers[col].str.strip().nunique(), '/', customers[col].nunique())

for col in ['payment_method', 'order_status']:
    print(col, ':', orders[col].str.strip().nunique(), '/', orders[col].nunique())

print('category', ':', products['category'].str.strip().nunique(), '/', products['category'].nunique())

gender : 2 / 2
city : 12 / 12
payment_method : 5 / 5
order_status : 6 / 6
category : 10 / 10


분석에 필수적인 파생 컬럼

In [14]:
order_items['revenue'] = order_items['quantity'] * order_items['unit_price']

valid_status = ['배송중', '배송완료', '결제완료', '배송준비']
orders['is_valid'] = orders['order_status'].isin(valid_status)

In [15]:
master = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(customers, on='customer_id', how='left')
)

print(master.shape)

(14603, 19)


In [16]:
# 병합 후 행 개수가 원본 order_items와 같아야 정상 (1:N 관계가 예상과 다르면 행이 뻥튀기됨)
print("병합 전 order_items 행 수:", len(order_items))
print("병합 후 master 행 수:", len(master))
assert len(master) == len(order_items), "병합 중 행 개수가 달라짐 — FK 관계 재확인 필요"

# 병합 후 결측치 발생 여부 확인 (조인 실패한 행이 있는지)
print(master[['order_status', 'category', 'gender']].isna().sum())

병합 전 order_items 행 수: 14603
병합 후 master 행 수: 14603
order_status    0
category        0
gender          0
dtype: int64


병합 결과

In [17]:
master.head()
master.dtypes

order_item_id              int64
order_id                   int64
product_id                 int64
quantity                   int64
unit_price                 int64
revenue                    int64
customer_id                int64
order_date        datetime64[us]
payment_method               str
order_status                 str
is_valid                    bool
product_name                 str
category                     str
price                      int64
name                         str
gender                       str
age                        int64
city                         str
signup_date       datetime64[us]
dtype: object

매출 관련 파생 컬럼

In [18]:
# 품목별 매출액
master['revenue'] = master['quantity'] * master['unit_price']

# 유효 주문 여부 (취소/환불 제외)
valid_status = ['배송중', '배송완료', '결제완료', '배송준비']
master['is_valid'] = master['order_status'].isin(valid_status)

# 확인
master[['revenue', 'is_valid']].head()

,revenue,is_valid
0,54400,True
1,87000,True
2,93000,True
3,225000,True
4,64000,False


시간 관련 파생 컬럼

In [19]:
master['order_month'] = master['order_date'].dt.to_period('M')
master['order_year'] = master['order_date'].dt.year
master['order_weekday'] = master['order_date'].dt.day_name()

고객 관련 파생 컬럼

In [20]:
master['age_group'] = (master['age'] // 10) * 10  # 10대, 20대, 30대 단위로 묶기

# 고객 가입 후 몇 개월 만에 이 주문을 했는지
master['months_since_signup'] = (
    (master['order_date'] - master['signup_date']).dt.days // 30
)

검증

In [21]:
print(master['revenue'].describe())
print(master['age_group'].value_counts().sort_index())
print(master['months_since_signup'].min(), master['months_since_signup'].max())

count    1.460300e+04
mean     1.258491e+05
std      1.418982e+05
min      3.400000e+03
25%      3.870000e+04
50%      8.100000e+04
75%      1.560000e+05
max      1.485000e+06
Name: revenue, dtype: float64
age_group
10     661
20    3134
30    5606
40    3971
50     944
60     287
Name: count, dtype: int64
0 30


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'src'))

from data_loader import load_all_data
customers, orders, order_items, products = load_all_data()